In [ ]:
# Celda 1
# Paths and parameters:
%cd ..

OUT_DIR = "data/fachadas"
RESOLUTION = 0.025   # Meters per pixel of the rectified images

# Facade coordinates: u = meters along the Toledo facade line, from the corner with Calle de los Estudios
# (see NOTES.md); z = height above sea level (same as the point cloud).
# Each job: photo, output name, reference points (photo pixel (x, y) -> facade point (u, z)),
# window to rectify (u_left > u_right: the photos look at the facade from the street, north on the left),
# and the sidewalk height used for the labels of the grid.
JOBS = [
    {
        "photo": "sources/imagenes/wikimedia/Instituto de San Isidro (Madrid) 01.jpg",
        "name": "toledo_instituto_u41-80",
        # A and B: meeting with the Colegiata; C and D: bend of the facade. D is hidden by people and was
        # computed as the crossing of the base line with the vertical of the bend [ESTIMADO].
        "references": [
            ((669, 67), (79.9, 661.0)),     # A: eave (tile edge)
            ((377, 3080), (79.9, 644.4)),   # B: sidewalk (moved +112 px so that window columns are vertical)
            ((3600, 1500), (40.9, 661.0)),  # C: eave (tile edge)
            ((3823, 2788), (40.9, 644.5)),  # D: sidewalk (moved +70 px so that window columns are vertical)
        ],
        "u_left": 80.5, "u_right": 40.5, "z_top": 662.5, "z_bottom": 644.0, "sidewalk": 644.4,
    },
    {
        "photo": "sources/imagenes/wikimedia/Main facade of the Colegiata de San Isidro 2022.jpg",
        "name": "toledo_colegiata_u80-111",
        # Frontal photo with verticals already corrected, but with different scales in x and y,
        # so the 4 points are the corners of a frame: facade edges (Catastro) x sidewalk / balustrade (cloud).
        "references": [
            ((604, 8428), (111.3, 644.4)),   # North edge (next to Toledo 35), sidewalk
            ((8030, 8428), (79.9, 644.4)),   # South edge (meeting with the Instituto), sidewalk
            ((604, 4286), (111.3, 663.5)),   # North edge, top of the balustrade rail
            ((8030, 4286), (79.9, 663.5)),   # South edge, top of the balustrade rail
        ],
        "u_left": 112.0, "u_right": 79.2, "z_top": 684.5, "z_bottom": 644.0, "sidewalk": 644.4,
    },
]

In [ ]:
# Celda 2
import os
import numpy as np
from PIL import Image, ImageDraw

Image.MAX_IMAGE_PIXELS = None   # The Colegiata photo has 75 million pixels
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Celda 3
# Pixel of the rectified image for a facade point:
# (u = u_left, z = z_top) -> (0, 0) ; one pixel = RESOLUTION meters
def to_rectified(job, u, z):
    column = (job["u_left"] - u) / RESOLUTION
    row = (job["z_top"] - z) / RESOLUTION
    return column, row

In [ ]:
# Celda 4
# Perspective transform (homography) from rectified pixels to photo pixels, with 8 unknowns a..h:
#   x = (a*c + b*r + c0) / (g*c + h*r + 1)
#   y = (d*c + e*r + f)  / (g*c + h*r + 1)
# Each reference point gives two linear equations, so 4 points give the 8 equations we need.
def homography(job):
    equations = []
    values = []
    for (x, y), (u, z) in job["references"]:
        c, r = to_rectified(job, u, z)
        equations.append([c, r, 1, 0, 0, 0, -c * x, -r * x])
        values.append(x)
        equations.append([0, 0, 0, c, r, 1, -c * y, -r * y])
        values.append(y)
    return np.linalg.solve(np.array(equations), np.array(values))

In [ ]:
# Celda 5
# Grid every meter on a copy of the rectified image (labels: u along the bottom, height above the sidewalk
# on the left), used to read the positions of windows, cornices and doors:
def draw_grid(job, image):
    width, height = image.size
    grid = image.copy()
    draw = ImageDraw.Draw(grid)
    for u in range(int(np.ceil(job["u_right"])), int(np.floor(job["u_left"])) + 1):
        c, r = to_rectified(job, u, job["z_bottom"])
        color = "red" if u % 5 == 0 else "yellow"
        draw.line([(c, 0), (c, height)], fill=color, width=1)
        if u % 5 == 0:
            draw.text((c + 3, height - 14), "u=" + str(u), fill="red")
    for z in range(int(np.ceil(job["z_bottom"])), int(np.floor(job["z_top"])) + 1):
        c, r = to_rectified(job, job["u_left"], z)
        color = "red" if z % 5 == 0 else "yellow"
        draw.line([(0, r), (width, r)], fill=color, width=1)
        draw.text((3, r + 2), str(round(z - job["sidewalk"], 1)) + " m", fill=color)
    return grid

In [ ]:
# Celda 6
# Rectify every photo and save the image and its grid copy:
for job in JOBS:
    width = int(round((job["u_left"] - job["u_right"]) / RESOLUTION))
    height = int(round((job["z_top"] - job["z_bottom"]) / RESOLUTION))
    coefficients = homography(job)
    photo = Image.open(job["photo"])
    rectified = photo.transform((width, height), Image.Transform.PERSPECTIVE, tuple(coefficients), Image.Resampling.BICUBIC)
    rectified.save(OUT_DIR + "/" + job["name"] + ".jpg", quality=92)
    draw_grid(job, rectified).save(OUT_DIR + "/" + job["name"] + "_grilla.jpg", quality=92)
    print(job["name"], "|", width, "x", height, "px ->", OUT_DIR)